In [1]:
from PIL import Image, ImageDraw, ImageFont
import koreanize_matplotlib

In [8]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

# 제목용 한글
import matplotlib
import matplotlib.pyplot as plt
import koreanize_matplotlib


# =====================================
# 경로 설정
# =====================================
ROOT = Path(r"C:\Users\hwapyeong\Documents\computer_vision\Rating_Batch_comparison\Rating_Batch_comparison")

DEBUG_ROOT = ROOT / "debug_crop_check"
OUT_ROOT = ROOT / "prediction_compare_outputs_v2"
OUT_ROOT.mkdir(exist_ok=True)

MODEL_PATHS = {
    "YOLO11x": ROOT / "yolo11x-seg.pt",
    "YOLO11_bh": ROOT / "best_FT_ epoch10_bh.pt",   # 공백 포함
}

PRED_CONF = 0.10

# 패널 표시 크기 통일
DISPLAY_HEIGHT = 1100   # 모든 결과 패널 높이를 동일하게 맞춤
PANEL_GAP = 30
TITLE_HEIGHT = 110
MARGIN = 30

# 모델 색상 (RGB)
MODEL_COLORS = {
    "YOLO11x": (255, 140, 0),    # orange
    "YOLO11_bh": (160, 32, 240), # purple
}

# 표용 한글 클래스명
CLASS_NAME_KR = {
    "book": "책",
    "bottle": "병",
    "hair drier": "헤어드라이어",
    "hair dryer": "헤어드라이어",
    "handbag": "가방",
    "keyboard": "키보드",
    "mouse": "마우스",
    "scissors": "가위",
    "toothbrush": "칫솔",
    "cell phone": "휴대폰",
    "laptop": "노트북",
}


# =====================================
# 기본 유틸
# =====================================
def load_rgb_image(img_path):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {img_path}")
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def get_input_image_path(image_name):
    image_dir = DEBUG_ROOT / image_name / "00_model_input"
    candidates = sorted(image_dir.glob("*.jpg"))
    if not candidates:
        raise FileNotFoundError(f"입력 이미지를 찾을 수 없습니다: {image_dir}")
    return candidates[0]


def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]


def overlay_mask(image_rgb, mask, color=(255, 0, 0), alpha=0.32):
    out = image_rgb.copy().astype(np.float32)
    color_arr = np.array(color, dtype=np.float32)
    mask_bool = mask.astype(bool)
    out[mask_bool] = out[mask_bool] * (1 - alpha) + color_arr * alpha
    return out.astype(np.uint8)


def resize_image_and_masks(image_rgb, pred_instances, target_height=1100):
    """
    표시용으로 이미지와 마스크를 같은 비율로 resize
    """
    h, w = image_rgb.shape[:2]
    scale = target_height / h
    new_w = int(round(w * scale))
    new_h = target_height

    resized_img = cv2.resize(image_rgb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    resized_instances = []
    for ins in pred_instances:
        mask = ins["mask"].astype(np.uint8)
        resized_mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
        resized_mask = (resized_mask > 0).astype(np.uint8)

        resized_instances.append({
            "class_name": ins["class_name"],
            "score": ins["score"],
            "mask": resized_mask,
            "bbox": mask_to_bbox(resized_mask),
        })

    return resized_img, resized_instances


def get_draw_style(image_h, image_w):
    """
    표시용 크기 기준으로 라벨 크기 통일
    """
    # DISPLAY_HEIGHT가 같기 때문에 대부분 거의 동일한 크기 느낌으로 출력됨
    font_scale = max(1.8, min(2.2, image_h / 500))
    thickness = max(4, int(round(image_h / 260)))
    contour_thickness = max(3, int(round(image_h / 320)))
    pad_x = max(10, int(round(image_h / 90)))
    pad_y = max(8, int(round(image_h / 120)))

    return {
        "font": cv2.FONT_HERSHEY_SIMPLEX,
        "font_scale": font_scale,
        "thickness": thickness,
        "contour_thickness": contour_thickness,
        "pad_x": pad_x,
        "pad_y": pad_y,
    }

def boxes_intersect(box1, box2):
    """
    box = (x1, y1, x2, y2)
    """
    ax1, ay1, ax2, ay2 = box1
    bx1, by1, bx2, by2 = box2

    if ax2 < bx1 or bx2 < ax1:
        return False
    if ay2 < by1 or by2 < ay1:
        return False
    return True


def clamp_box_to_image(box, image_w, image_h):
    x1, y1, x2, y2 = box
    box_w = x2 - x1
    box_h = y2 - y1

    x1 = max(0, min(x1, image_w - box_w - 1))
    y1 = max(0, min(y1, image_h - box_h - 1))
    x2 = x1 + box_w
    y2 = y1 + box_h
    return (x1, y1, x2, y2)


def find_non_overlapping_label_position(
    bbox,
    text_size,
    image_w,
    image_h,
    pad_x,
    pad_y,
    occupied_boxes
):
    """
    bbox: [x1, y1, x2, y2]
    text_size: (tw, th, baseline)
    occupied_boxes: 이미 사용한 라벨 박스 목록
    """
    x1, y1, x2, y2 = bbox
    tw, th, baseline = text_size

    label_w = tw + pad_x * 2
    label_h = th + baseline + pad_y * 2

    # 후보 위치들: 위, 아래, 오른쪽 위, 왼쪽 위, 오른쪽 아래, 왼쪽 아래
    candidates = [
        (x1, y1 - label_h - 10),          # 위
        (x1, y1 + 10),                    # 아래
        (x2 + 10, y1),                    # 오른쪽
        (x2 + 10, y2 - label_h),          # 오른쪽 아래
        (x1 - label_w - 10, y1),          # 왼쪽
        (x1 - label_w - 10, y2 - label_h) # 왼쪽 아래
    ]

    # 우선순위대로 안 겹치는 위치 찾기
    for cx, cy in candidates:
        box = (cx, cy, cx + label_w, cy + label_h)
        box = clamp_box_to_image(box, image_w, image_h)

        overlapped = any(boxes_intersect(box, used) for used in occupied_boxes)
        if not overlapped:
            return box

    # 그래도 겹치면 아래로 조금씩 내려가며 탐색
    start_x = max(0, min(x1, image_w - label_w - 1))
    for offset in range(0, image_h, max(20, label_h // 3)):
        cy = min(max(0, y1 + offset), max(0, image_h - label_h - 1))
        box = (start_x, cy, start_x + label_w, cy + label_h)
        overlapped = any(boxes_intersect(box, used) for used in occupied_boxes)
        if not overlapped:
            return box

    # 끝까지 못 찾으면 기본 위치 반환
    box = (x1, max(0, y1 - label_h - 10), x1 + label_w, max(0, y1 - label_h - 10) + label_h)
    box = clamp_box_to_image(box, image_w, image_h)
    return box


def draw_contour_and_label(image_rgb, mask, label_text, color=(255, 0, 0), score=None, occupied_boxes=None):
    """
    영어 라벨 전용.
    occupied_boxes: 이미 배치한 라벨 박스 목록
    """
    if occupied_boxes is None:
        occupied_boxes = []

    img_bgr = cv2.cvtColor(image_rgb.copy(), cv2.COLOR_RGB2BGR)
    mask_u8 = (mask > 0).astype(np.uint8) * 255

    h, w = image_rgb.shape[:2]
    style = get_draw_style(h, w)

    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(
        img_bgr,
        contours,
        -1,
        color,
        style["contour_thickness"]
    )

    bbox = mask_to_bbox(mask)
    if bbox is not None:
        text = label_text if score is None else f"{label_text} {score:.2f}"

        (tw, th), baseline = cv2.getTextSize(
            text,
            style["font"],
            style["font_scale"],
            style["thickness"]
        )

        label_box = find_non_overlapping_label_position(
            bbox=bbox,
            text_size=(tw, th, baseline),
            image_w=w,
            image_h=h,
            pad_x=style["pad_x"],
            pad_y=style["pad_y"],
            occupied_boxes=occupied_boxes
        )

        box_x1, box_y1, box_x2, box_y2 = label_box

        cv2.rectangle(img_bgr, (box_x1, box_y1), (box_x2, box_y2), color, -1)

        text_x = box_x1 + style["pad_x"]
        text_y = box_y2 - style["pad_y"] - baseline

        cv2.putText(
            img_bgr,
            text,
            (text_x, text_y),
            style["font"],
            style["font_scale"],
            (255, 255, 255),
            style["thickness"],
            cv2.LINE_AA
        )

        occupied_boxes.append(label_box)

    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), occupied_boxes


def run_model_prediction(model, image_path, conf=0.10):
    results = model.predict(
        source=str(image_path),
        conf=conf,
        verbose=False
    )

    pred_instances = []
    if len(results) == 0:
        return pred_instances

    r = results[0]

    if r.masks is None or r.boxes is None:
        return pred_instances

    orig_h, orig_w = r.orig_shape

    masks = r.masks.data.cpu().numpy()
    cls_ids = r.boxes.cls.cpu().numpy().astype(int)
    scores = r.boxes.conf.cpu().numpy()
    names = r.names

    for mask, cls_id, score in zip(masks, cls_ids, scores):
        class_name = names[int(cls_id)]
        mask = (mask > 0.5).astype(np.uint8)

        # 원본 크기로 복원
        if mask.shape != (orig_h, orig_w):
            mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            mask = (mask > 0).astype(np.uint8)

        bbox = mask_to_bbox(mask)
        if bbox is None:
            continue

        pred_instances.append({
            "class_name": class_name,
            "score": float(score),
            "mask": mask,
            "bbox": bbox
        })

    return pred_instances

def make_prediction_overlay(image_rgb, pred_instances, color_rgb):
    """
    1) 먼저 표시 크기 통일
    2) 그 위에 마스크/라벨 그림
    3) 라벨끼리 안 겹치도록 순차 배치
    """
    display_img, display_preds = resize_image_and_masks(
        image_rgb,
        pred_instances,
        target_height=DISPLAY_HEIGHT
    )

    # 큰 객체부터 먼저 라벨 배치하면 겹침이 덜함
    def mask_area(ins):
        return int(ins["mask"].sum())

    display_preds = sorted(display_preds, key=mask_area, reverse=True)

    canvas = display_img.copy()
    occupied_boxes = []

    for ins in display_preds:
        canvas = overlay_mask(canvas, ins["mask"], color=color_rgb, alpha=0.25)
        canvas, occupied_boxes = draw_contour_and_label(
            canvas,
            ins["mask"],
            ins["class_name"],   # 영어 라벨 유지
            color=color_rgb,
            score=ins["score"],
            occupied_boxes=occupied_boxes
        )

    return canvas


def add_title_band(panel_rgb, title_text):
    """
    패널 위에 제목 영역 추가
    제목은 cv2 영어/숫자 기반으로도 충분하지만,
    한글 제목을 안정적으로 넣기 위해 matplotlib로 작은 그림 생성 후 붙임
    """
    h, w = panel_rgb.shape[:2]

    fig = plt.figure(figsize=(w / 100, TITLE_HEIGHT / 100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")
    ax.text(
        0.5, 0.5, title_text,
        ha="center", va="center",
        fontsize=28, fontweight="bold"
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    fig.canvas.draw()
    title_img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    title_img = title_img.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    title_img = title_img[:, :, :3]
    plt.close(fig)

    title_img = cv2.resize(title_img, (w, TITLE_HEIGHT), interpolation=cv2.INTER_LINEAR)

    out = np.full((h + TITLE_HEIGHT, w, 3), 255, dtype=np.uint8)
    out[:TITLE_HEIGHT] = title_img
    out[TITLE_HEIGHT:] = panel_rgb
    return out


def hstack_panels_equal_height(panels, gap=30, bg_color=255):
    heights = [p.shape[0] for p in panels]
    max_h = max(heights)

    padded = []
    for p in panels:
        h, w = p.shape[:2]
        if h < max_h:
            pad = np.full((max_h - h, w, 3), bg_color, dtype=np.uint8)
            p = np.vstack([p, pad])
        padded.append(p)

    total_w = sum(p.shape[1] for p in padded) + gap * (len(padded) - 1)
    canvas = np.full((max_h, total_w, 3), bg_color, dtype=np.uint8)

    x = 0
    for p in padded:
        h, w = p.shape[:2]
        canvas[:h, x:x+w] = p
        x += w + gap

    return canvas


def predictions_to_korean_rows(image_name, model_name, pred_instances):
    rows = []

    if len(pred_instances) == 0:
        rows.append({
            "이미지": image_name,
            "모델": model_name,
            "예측 개수": 0,
            "순번": "-",
            "예측 라벨": "-",
            "한글 설명": "-",
            "신뢰도": "-"
        })
        return rows

    for idx, ins in enumerate(pred_instances, start=1):
        eng = ins["class_name"]
        kor = CLASS_NAME_KR.get(eng, eng)

        rows.append({
            "이미지": image_name,
            "모델": model_name,
            "예측 개수": len(pred_instances),
            "순번": idx,
            "예측 라벨": eng,
            "한글 설명": kor,
            "신뢰도": round(ins["score"], 4)
        })

    return rows


def save_prediction_comparison(image_name, models):
    img_path = get_input_image_path(image_name)
    image_rgb = load_rgb_image(img_path)

    model_names = list(models.keys())
    all_rows = []

    # 원본도 표시 크기 통일
    base_display, _ = resize_image_and_masks(image_rgb, [], target_height=DISPLAY_HEIGHT)

    overlay_results = {}
    pred_results = {}

    for model_name in model_names:
        pred_instances = run_model_prediction(models[model_name], img_path, conf=PRED_CONF)
        pred_results[model_name] = pred_instances

        overlay_results[model_name] = make_prediction_overlay(
            image_rgb,
            pred_instances,
            MODEL_COLORS[model_name]
        )

        all_rows.extend(predictions_to_korean_rows(image_name, model_name, pred_instances))

    # 제목 붙이기
    panel_0 = add_title_band(base_display, "원본 이미지")
    panel_1 = add_title_band(overlay_results[model_names[0]], f"{model_names[0]} 예측 결과")
    panel_2 = add_title_band(overlay_results[model_names[1]], f"{model_names[1]} 예측 결과")

    combined = hstack_panels_equal_height([panel_0, panel_1, panel_2], gap=PANEL_GAP)

    save_path = OUT_ROOT / f"{image_name}_prediction_compare.png"
    combined_bgr = cv2.cvtColor(combined, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(save_path), combined_bgr)

    print(f"[저장 완료] {save_path}")

    df = pd.DataFrame(all_rows)
    return df


def main():
    print("=== 모델 확인 ===")
    for model_name, model_path in MODEL_PATHS.items():
        print(model_name, "->", model_path, "| exists:", model_path.exists())

    models = {}
    for model_name, model_path in MODEL_PATHS.items():
        if not model_path.exists():
            raise FileNotFoundError(f"모델 파일이 없습니다: {model_path}")
        models[model_name] = YOLO(str(model_path))
        print(f"[로드 완료] {model_name}")

    target_images = [f"image_{i:03d}" for i in range(1, 7)]

    total_df_list = []

    for image_name in target_images:
        print(f"\n[처리 중] {image_name}")
        df = save_prediction_comparison(image_name, models)
        total_df_list.append(df)

    final_df = pd.concat(total_df_list, ignore_index=True)

    csv_path = OUT_ROOT / "prediction_summary_ko.csv"
    final_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"\n[CSV 저장 완료] {csv_path}")
    return final_df

In [9]:
result_df = main()
result_df

=== 모델 확인 ===
YOLO11x -> C:\Users\hwapyeong\Documents\computer_vision\Rating_Batch_comparison\Rating_Batch_comparison\yolo11x-seg.pt | exists: True
YOLO11_bh -> C:\Users\hwapyeong\Documents\computer_vision\Rating_Batch_comparison\Rating_Batch_comparison\best_FT_ epoch10_bh.pt | exists: True
[로드 완료] YOLO11x
[로드 완료] YOLO11_bh

[처리 중] image_001
[저장 완료] C:\Users\hwapyeong\Documents\computer_vision\Rating_Batch_comparison\Rating_Batch_comparison\prediction_compare_outputs_v2\image_001_prediction_compare.png

[처리 중] image_002
[저장 완료] C:\Users\hwapyeong\Documents\computer_vision\Rating_Batch_comparison\Rating_Batch_comparison\prediction_compare_outputs_v2\image_002_prediction_compare.png

[처리 중] image_003
[저장 완료] C:\Users\hwapyeong\Documents\computer_vision\Rating_Batch_comparison\Rating_Batch_comparison\prediction_compare_outputs_v2\image_003_prediction_compare.png

[처리 중] image_004
[저장 완료] C:\Users\hwapyeong\Documents\computer_vision\Rating_Batch_comparison\Rating_Batch_comparison\predictio

,이미지,모델,예측 개수,순번,예측 라벨,한글 설명,신뢰도
0,image_001,YOLO11x,6,1,keyboard,키보드,0.9629
1,image_001,YOLO11x,6,2,handbag,가방,0.9219
2,image_001,YOLO11x,6,3,cell phone,휴대폰,0.6629
3,image_001,YOLO11x,6,4,toothbrush,칫솔,0.6441
4,image_001,YOLO11x,6,5,book,책,0.4442
...,...,...,...,...,...,...,...
75,image_006,YOLO11_bh,6,2,toothbrush,칫솔,0.6459
76,image_006,YOLO11_bh,6,3,cell phone,휴대폰,0.2610
77,image_006,YOLO11_bh,6,4,book,책,0.2262
78,image_006,YOLO11_bh,6,5,keyboard,키보드,0.1655
